In [1]:
# Standard Imports
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import tensorflow as tf

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

I0000 00:00:1778772131.832744    8373 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1778772131.874995    8373 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1778772132.806151    8373 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
# Dataset import
housing = fetch_california_housing()
X_train_full, X_test, y_train_full, y_test = train_test_split(
    housing.data, housing.target, random_state=13)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full, y_train_full, random_state=13)

In [3]:
X_train_wide, X_train_deep = X_train[:, :5], X_train[:, 2:]
X_valid_wide, X_valid_deep = X_valid[:, :5], X_valid[:, 2:]
X_test_wide, X_test_deep = X_test[:, :5], X_test[:, 2:]
X_new_wide, X_new_deep = X_test_wide[:3], X_test_deep[:3]

# Lab 7.9 Building Complex Models - Using Classes and Design Patterns

## The Wide & Deep neural network

In [ ]:
tf.random.set_seed(13)

normalization_layer = tf.keras.layers.Normalization()
hidden_layer1 = tf.keras.layers.Dense(30, activation="relu")
hidden_layer2 = tf.keras.layers.Dense(30, activation="relu")
concat_layer = tf.keras.layers.Concatenate()
output_layer = tf.keras.layers.Dense(1)

input_ = tf.keras.layers.Input(shape=X_train.shape[1:])
normalized = normalization_layer(input_)  # split to wide
hidden1 = hidden_layer1(normalized)
hidden2 = hidden_layer2(hidden1)
concat = concat_layer([normalized, hidden2])   # bring in wide
output = output_layer(concat)

model = tf.keras.Model(inputs=[input_], outputs=[output])

## Subsetting data through the network

In [8]:
input_wide = tf.keras.layers.Input(shape=[5])  # features 0 to 4
input_deep = tf.keras.layers.Input(shape=[6])  # features 2 to 7
norm_layer_wide = tf.keras.layers.Normalization()
norm_layer_deep = tf.keras.layers.Normalization()
norm_wide = norm_layer_wide(input_wide)
norm_deep = norm_layer_deep(input_deep)

hidden1 = tf.keras.layers.Dense(30, activation="relu")(norm_deep)
hidden2 = tf.keras.layers.Dense(30, activation="relu")(hidden1)
concat = tf.keras.layers.concatenate([norm_wide, hidden2])
output = tf.keras.layers.Dense(1)(concat)
model = tf.keras.Model(inputs=[input_wide, input_deep], outputs=[output])

E0000 00:00:1778718483.572476    6512 cuda_executor.cc:1737] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1778718483.572695    7044 cuda_executor.cc:1755] Failed to determine cuDNN version (Note that this is expected if the application doesn't link the cuDNN plugin): INTERNAL: cuDNN error: CUDNN_STATUS_INTERNAL_ERROR
W0000 00:00:1778718483.585458    6512 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


In [9]:
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3)
model.compile(loss="mse", optimizer=optimizer, metrics=["RootMeanSquaredError"])

X_train_wide, X_train_deep = X_train[:, :5], X_train[:, 2:]
X_valid_wide, X_valid_deep = X_valid[:, :5], X_valid[:, 2:]
X_test_wide, X_test_deep = X_test[:, :5], X_test[:, 2:]
X_new_wide, X_new_deep = X_test_wide[:3], X_test_deep[:3]

norm_layer_wide.adapt(X_train_wide)
norm_layer_deep.adapt(X_train_deep)
history = model.fit((X_train_wide, X_train_deep), y_train, epochs=20,
                    validation_data=((X_valid_wide, X_valid_deep), y_valid))
mse_test = model.evaluate((X_test_wide, X_test_deep), y_test)
y_pred = model.predict((X_new_wide, X_new_deep))

Epoch 1/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - RootMeanSquaredError: 1.1501 - loss: 1.3228 - val_RootMeanSquaredError: 0.7744 - val_loss: 0.5997
Epoch 2/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - RootMeanSquaredError: 0.7409 - loss: 0.5490 - val_RootMeanSquaredError: 0.6590 - val_loss: 0.4343
Epoch 3/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - RootMeanSquaredError: 0.6657 - loss: 0.4432 - val_RootMeanSquaredError: 0.6226 - val_loss: 0.3877
Epoch 4/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - RootMeanSquaredError: 0.6396 - loss: 0.4091 - val_RootMeanSquaredError: 0.6066 - val_loss: 0.3680
Epoch 5/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - RootMeanSquaredError: 0.6249 - loss: 0.3906 - val_RootMeanSquaredError: 0.6004 - val_loss: 0.3604
Epoch 6/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - RootMeanSquaredError: 0.6284 - loss: 0.3949 - val_RootMeanSquaredError: 0.6168 - val_loss: 0.3804
Epoch 7/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - RootMeanSquaredError: 0.6221 - los

## Using Classes

In [4]:
# extra code – clear the session to reset the name counters
tf.keras.backend.clear_session()
tf.random.set_seed(13)

Create a class for doing wide and deep design model expirements.

In [5]:
class WideAndDeepModel(tf.keras.Model):
    def __init__(self, layer_sizes=[30, 30], activation="relu", **kwargs):
        super().__init__(**kwargs)
        self.norm_layer_wide = tf.keras.layers.Normalization()
        self.norm_layer_deep = tf.keras.layers.Normalization()
        self.hidden_layers = [
            tf.keras.layers.Dense(size, activation=activation)
            for size in layer_sizes
        ]
        self.main_output = tf.keras.layers.Dense(1)

    def call(self, inputs):
        input_wide, input_deep = inputs
        norm_wide = self.norm_layer_wide(input_wide)
        norm_deep = self.norm_layer_deep(input_deep)
        deep = norm_deep
        for layer in self.hidden_layers:
            deep = layer(deep)
        concat = tf.keras.layers.concatenate([norm_wide, deep])
        output = self.main_output(concat)
        return output

## Wide and deep with defaults of class (2 layers, 30 nuerons each, relu activation)

In [9]:
# Instantiate the model
default_model = WideAndDeepModel()

# Adapt normalisation layers
default_model.norm_layer_wide.adapt(X_train_wide)
default_model.norm_layer_deep.adapt(X_train_deep)

# Compile
default_model.compile(optimizer=tf.keras.optimizers.Adam(0.0001),
                      loss="mse",
                      metrics=["RootMeanSquaredError"])

# 4. Fit
import datetime

log_dir = "logs/fit/default_model_" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")

history = default_model.fit(
    (X_train_wide, X_train_deep),
    y_train,
    epochs=100,
    validation_data=((X_valid_wide, X_valid_deep), y_valid),
    callbacks=[
        tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
        tf.keras.callbacks.TensorBoard(log_dir=log_dir)
    ]
)

Epoch 1/100
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - RootMeanSquaredError: 2.1465 - loss: 4.6073 - val_RootMeanSquaredError: 1.7744 - val_loss: 3.1486
Epoch 2/100
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - RootMeanSquaredError: 1.6296 - loss: 2.6555 - val_RootMeanSquaredError: 1.3305 - val_loss: 1.7701
Epoch 3/100
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - RootMeanSquaredError: 1.3155 - loss: 1.7306 - val_RootMeanSquaredError: 1.1341 - val_loss: 1.2861
Epoch 4/100
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - RootMeanSquaredError: 1.1640 - loss: 1.3549 - val_RootMeanSquaredError: 1.0481 - val_loss: 1.0985
Epoch 5/100
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - RootMeanSquaredError: 1.0762 - loss: 1.1582 - val_RootMeanSquaredError: 0.9915 - val_loss: 0.9830
Epoch 6/100
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - RootMeanSquaredError: 1.0106 - loss: 1.0213 - val_RootMeanSquaredError: 0.9445 - val_loss: 0.8922
Epoch 7/100
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - RootMeanSquaredError: 0.957

## Wide and deep with more layers

In [10]:
# Instantiate the model
more_layers_model = WideAndDeepModel(layer_sizes=[30,30,30,30,30])

# Adapt normalisation layers
more_layers_model.norm_layer_wide.adapt(X_train_wide)
more_layers_model.norm_layer_deep.adapt(X_train_deep)

# Compile
more_layers_model.compile(optimizer=tf.keras.optimizers.Adam(0.0001),
                      loss="mse",
                      metrics=["RootMeanSquaredError"])

# 4. Fit
log_dir = "logs/fit/more_layers_model_" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")

history = more_layers_model.fit(
    (X_train_wide, X_train_deep),
    y_train,
    epochs=100,
    validation_data=((X_valid_wide, X_valid_deep), y_valid),
    callbacks=[
        tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
        tf.keras.callbacks.TensorBoard(log_dir=log_dir)
    ]
)

Epoch 1/100
363/363 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - RootMeanSquaredError: 1.9467 - loss: 3.7896 - val_RootMeanSquaredError: 1.3024 - val_loss: 1.6963
Epoch 2/100
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - RootMeanSquaredError: 1.2608 - loss: 1.5896 - val_RootMeanSquaredError: 1.0781 - val_loss: 1.1624
Epoch 3/100
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - RootMeanSquaredError: 1.0371 - loss: 1.0757 - val_RootMeanSquaredError: 0.9297 - val_loss: 0.8644
Epoch 4/100
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - RootMeanSquaredError: 0.9273 - loss: 0.8599 - val_RootMeanSquaredError: 0.8787 - val_loss: 0.7720
Epoch 5/100
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - RootMeanSquaredError: 0.8850 - loss: 0.7833 - val_RootMeanSquaredError: 0.8483 - val_loss: 0.7197
Epoch 6/100
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - RootMeanSquaredError: 0.8563 - loss: 0.7333 - val_RootMeanSquaredError: 0.8231 - val_loss: 0.6775
Epoch 7/100
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - RootMeanSquaredError: 0.832

## Wide and deep with more neurons

In [11]:
# Instantiate the model
more_neurons_model = WideAndDeepModel(layer_sizes=[90,90])

# Adapt normalisation layers
more_neurons_model.norm_layer_wide.adapt(X_train_wide)
more_neurons_model.norm_layer_deep.adapt(X_train_deep)

# Compile
more_neurons_model.compile(optimizer=tf.keras.optimizers.Adam(0.0001),
                      loss="mse",
                      metrics=["RootMeanSquaredError"])

# 4. Fit
log_dir = "logs/fit/more_neurons_model_" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")

history = more_neurons_model.fit(
    (X_train_wide, X_train_deep),
    y_train,
    epochs=100,
    validation_data=((X_valid_wide, X_valid_deep), y_valid),
    callbacks=[
        tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
        tf.keras.callbacks.TensorBoard(log_dir=log_dir)
    ]
)

Epoch 1/100
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - RootMeanSquaredError: 1.9583 - loss: 3.8348 - val_RootMeanSquaredError: 1.4646 - val_loss: 2.1452
Epoch 2/100
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - RootMeanSquaredError: 1.4393 - loss: 2.0714 - val_RootMeanSquaredError: 1.2726 - val_loss: 1.6196
Epoch 3/100
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - RootMeanSquaredError: 1.2507 - loss: 1.5642 - val_RootMeanSquaredError: 1.1310 - val_loss: 1.2791
Epoch 4/100
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - RootMeanSquaredError: 1.0924 - loss: 1.1934 - val_RootMeanSquaredError: 0.9973 - val_loss: 0.9947
Epoch 5/100
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - RootMeanSquaredError: 0.9651 - loss: 0.9314 - val_RootMeanSquaredError: 0.8956 - val_loss: 0.8021
Epoch 6/100
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - RootMeanSquaredError: 0.8806 - loss: 0.7754 - val_RootMeanSquaredError: 0.8335 - val_loss: 0.6948
Epoch 7/100
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - RootMeanSquaredError: 0.831

## Wide and Deep with Sigmoid Activation

In [12]:
# Instantiate the model
sigmoid_activation_model = WideAndDeepModel(layer_sizes=[30,30], activation="sigmoid")

# Adapt normalisation layers
sigmoid_activation_model.norm_layer_wide.adapt(X_train_wide)
sigmoid_activation_model.norm_layer_deep.adapt(X_train_deep)

# Compile
sigmoid_activation_model.compile(optimizer=tf.keras.optimizers.Adam(0.0001),
                      loss="mse",
                      metrics=["RootMeanSquaredError"])

# 4. Fit
log_dir = "logs/fit/sigmoid_activation_model_" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")

history = sigmoid_activation_model.fit(
    (X_train_wide, X_train_deep),
    y_train,
    epochs=100,
    validation_data=((X_valid_wide, X_valid_deep), y_valid),
    callbacks=[
        tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
        tf.keras.callbacks.TensorBoard(log_dir=log_dir)
    ]
)

Epoch 1/100
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - RootMeanSquaredError: 1.5672 - loss: 2.4561 - val_RootMeanSquaredError: 1.2027 - val_loss: 1.4466
Epoch 2/100
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - RootMeanSquaredError: 1.1276 - loss: 1.2716 - val_RootMeanSquaredError: 1.0606 - val_loss: 1.1249
Epoch 3/100
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - RootMeanSquaredError: 1.0622 - loss: 1.1282 - val_RootMeanSquaredError: 1.0333 - val_loss: 1.0677
Epoch 4/100
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - RootMeanSquaredError: 1.0358 - loss: 1.0730 - val_RootMeanSquaredError: 1.0089 - val_loss: 1.0179
Epoch 5/100
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - RootMeanSquaredError: 1.0116 - loss: 1.0233 - val_RootMeanSquaredError: 0.9857 - val_loss: 0.9715
Epoch 6/100
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - RootMeanSquaredError: 0.9888 - loss: 0.9777 - val_RootMeanSquaredError: 0.9638 - val_loss: 0.9288
Epoch 7/100
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - RootMeanSquaredError: 0.967

In [13]:
%tensorboard --logdir logs/fit

Reusing TensorBoard on port 6006 (pid 9716), started 0:23:27 ago. (Use '!kill 9716' to kill it.)

In [14]:
%kill 9719

UsageError: Line magic function `%kill` not found.
